In [4]:
# --- 1. Installation (MUST BE AT THE TOP) ---
import os
import sys

# Install necessary wheels before any imports
print("Installing Biopython and Biotite...")
os.system("pip install --no-index --no-deps /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl")
os.system("pip install --no-index --no-deps /kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl")

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from Bio.Align import PairwiseAligner # Should work now

# --- 2. Adaptive Physics Refiner (The Secret Sauce for Higher Scores) ---
def adaptive_rna_constraints(coords, confidence=0.8, passes=2):
    """
    Apply RNA physics constraints: C1'-C1' atoms should be ~5.95A apart.
    This step refines the Protenix output to be more 'physically correct'.
    """
    X = coords.copy()
    strength = max(0.6 * (1.0 - min(confidence, 0.95)), 0.05)
    
    for _ in range(passes):
        # 1. Distance constraint (i to i+1)
        d = X[1:] - X[:-1]
        dist = np.linalg.norm(d, axis=1, keepdims=True) + 1e-6
        adjustment = d * ((5.95 - dist) / dist) * (0.22 * strength)
        X[:-1] -= adjustment
        X[1:] += adjustment
        
        # 2. Angle/Volume constraint (i to i+2)
        if len(X) > 2:
            d2 = X[2:] - X[:-2]
            dist2 = np.linalg.norm(d2, axis=1, keepdims=True) + 1e-6
            adj2 = d2 * ((10.2 - dist2) / dist2) * (0.12 * strength)
            X[:-2] -= adj2
            X[2:] += adj2
    return X

# --- 3. Integration with your 0.426 Logic ---
def run_pipeline():
    print("Running Stanford_RNA_P2_Hybrid_TBM_Protenix_Refiner...")
    
    # Path setup based on your original environment
    DATA_BASE = Path("/kaggle/input/competitions/stanford-rna-3d-folding-2")
    TEST_CSV = DATA_BASE / "test_sequences.csv"
    
    if not TEST_CSV.exists():
        print("Test file not found. Running in local/test mode.")
        return

    test_df = pd.read_csv(TEST_CSV)
    id_col = 'target_id' # As per competition data
    
    # Note: In a real run, you should import the actual model inference 
    # from your stanford-rna-p2-3d-folding-lb0-433.ipynb here.
    # Below is the logic to save and refine.
    
    all_submission_rows = []

    for idx, row in test_df.iterrows():
        tid = row[id_col]
        seq = row['sequence']
        seq_len = len(seq)
        
        # --- MODEL PREDICTION STEP ---
        # Assuming you have loaded your model as 'model'
        # coords = model.predict(seq) # This should yield (5, seq_len, 3)
        
        # (Placeholder for this example: using a noisy helix to represent model output)
        # In your actual notebook, keep your 0.426 inference code here!
        predicted_coords = np.zeros((5, seq_len, 3)) 
        
        for m in range(5):
            # --- APPLY REFINER ---
            # We apply the 'Refiner' to each of the 5 predicted models
            refined = adaptive_rna_constraints(predicted_coords[m], confidence=0.8)
            predicted_coords[m] = refined

        # --- 4. Format to Submission CSV ---
        for i in range(seq_len):
            res_name = seq[i]
            res_id = i + 1
            row_id = f"{tid}_{res_id}"
            
            line = [row_id, res_name, res_id]
            for m in range(5):
                line.extend(predicted_coords[m][i])
            
            all_submission_rows.append(line)

    # Define Columns
    cols = ['ID', 'resname', 'resid']
    for m in range(1, 6):
        cols += [f'x_{m}', f'y_{m}', f'z_{m}']
    
    sub_df = pd.DataFrame(all_submission_rows, columns=cols)
    sub_df.to_csv("submission.csv", index=False)
    print("Submission file 'submission.csv' generated successfully.")

if __name__ == "__main__":
    run_pipeline()

Installing Biopython and Biotite...
Processing /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
Running Stanford_RNA_P2_Hybrid_TBM_Protenix_Refiner...
Submission file 'submission.csv' generated successfully.
